# 07 面向完成项目的微调流程

从这一课开始，以“学明白微调并完成口罩佩戴检测”为目标，不再继续展开 anchor、目标分配和损失公式。

你已具备足够的概念基础。接下来主要学习：准备什么数据、设置哪些参数、怎样判断训练结果，以及怎样把自己的权重接入网页。

## 1. 我们实际要完成五件事

| 步骤 | 要做什么 | 完成的标志 |
|---|---|---|
| 数据准备 | 检查标注、处理重复、转换标签、固定划分 | 图片与标签匹配，抽样画框正确 |
| 确定训练起点 | 使用兼容的预训练权重 | 明确来源，并确认加载成功 |
| 短程试跑 | 用少量训练轮次检查流程 | 能训练、验证、保存并重新加载权重 |
| 完整训练与检查 | 根据验证表现选择模型，查看误检漏检 | 留下指标与具体样例 |
| 接入部署 | 将训练得到的 best.pt 接入当前网页 | 图片、视频和摄像头流程可用 |

第一轮计划从通用 YOLOv5s 检测预训练权重开始，让模型学习三类口罩任务。已有作者口罩权重保留，方便对照；它可能见过现有数据，不能据此声称公平提升。

通用权重还没有下载，本轮也没有启动训练。

## 2. 训练前只需先认识这些参数

| 参数 | 直观含义 | 先怎样理解 |
|---|---|---|
| weights | 起始权重 | 从哪份已有经验开始学习 |
| data | 数据配置 | 去哪里读训练集、验证集，类别顺序是什么 |
| imgsz | 输入尺寸设置 | 例如 640；会影响细节、速度和显存开销 |
| batch | 每次前向处理的图片数 | 太大可能显存不足；不是越大效果一定越好 |
| epochs | 训练轮次 | 一轮通常表示训练集被遍历一次 |
| workers | 数据加载进程数 | Windows 第一轮可先用 0，减少并发与排错复杂度 |
| 学习率 | 参数更新幅度 | 先沿用合适的默认配置，出现明确问题再调 |

第一轮先不添加冻结设置，让模型按当前训练代码的默认方式微调。

可以先计划以输入尺寸 640、batch 4、workers 0，运行 3 轮来检查训练流程；这些只是待验证的起始设置，具体按实际内存、显存和兼容性调整。三轮试跑不等于已经充分训练。

## 3. 短程试跑与完整训练分开

短程试跑要回答的是：数据能否读取、GPU 能否训练、验证能否完成、权重能否保存和重新加载。

完整训练才主要回答：模型是否有效学习、验证表现如何、什么时候开始过拟合。

不用一开始就盲目运行 100 轮。先确认流程正确，再根据训练与验证曲线决定后续轮次。

开始训练前应检查可用内存与显存；网页推理服务和训练会竞争资源，因此正式运行时应先妥善停止已有服务。

## 4. 读结果只抓住三个重点

1. 训练损失是否总体合理下降：box、obj、cls 的含义分别是定位、目标存在性和类别，不必手算源码里的公式。
2. 验证指标是否支持模型有效：观察 Precision、Recall、mAP，并关注三类各自的表现；训练损失下降本身不能证明泛化变好。
3. 实际图片错在哪里：看漏掉的小脸、误报的背景、戴法分类错误等，尤其注意样本较少的“佩戴不正确”。

训练程序通常会保存 `best.pt` 与 `last.pt`：前者是按验证指标选择的权重，后者是最后一轮的权重。接入网页时先检查所选权重在实际样例上的效果。

新场景图片可用于直观检查；若没有人工标注，不能据此计算可靠的准确率或 mAP。

## 5. 当前项目的数据检查结果

本次直接读取压缩包，没有解压完整训练数据：

- 853 对图片与 XML 文件名对应。
- 三类标注框分别为：正确戴口罩 3232、未戴口罩 717、佩戴不正确 123。
- 有 5 组图片文件内容完全重复，每组 2 张。
- 有 12 个框超出图片右边界，涉及 11 张图片；相关图片实际宽度均为 400，标注 xmax 为 401。

重复图片组：

| 组 | 图片编号（省略共同前缀 maksssksksss） |
|---|---|
| 1 | 13、737 |
| 2 | 146、219 |
| 3 | 410、428 |
| 4 | 653、775 |
| 5 | 656、690 |

正式数据准备时，应让每组重复图片进入同一个子集，避免跨训练集、验证集泄漏；还需统一坐标约定、处理超界并抽样可视化。

本轮只发现问题，没有修改原始数据或运行旧转换脚本。由于需要按组划分，最终训练集与验证集的精确数量以实际划分记录为准，不预先承诺 682/171。

## 6. 下一步只做数据准备

接下来先将压缩包整理成 YOLO 可读的图片与标签目录，处理已发现的数据问题，保留可复现的划分记录，画出若干标签检查。

数据确认正确后，再进入短程训练，而不是继续增加理论课。

本课仅新增实践路线笔记。没有下载新权重、创建训练脚本或启动训练。

参考：[Ultralytics 官方自定义数据训练指南](https://docs.ultralytics.com/yolov5/tutorials/train-custom-data)。本地检查结果来自实际压缩包与代码，不代表训练效果。